# SeqFrame: sequence alignment as a DataFrame

In [22]:
%load_ext autoreload
%autoreload 2
import pandas as pd
from pdbframe import SeqFrame

seq1 = 'GSHSMRYFYTSVSRPGRGEPRFIAVGYVDDTQFVRF'
seq2 = 'SMRYFFTSVSRPGRGEPRFIAVGYVDDTQFVRFDSDAASQRMEPRAPW'

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Align
These two calls do the same thing. The result displays as the alignment table.

In [23]:
out = SeqFrame(seq1).align(seq2)
out = SeqFrame.align(seq1, seq2)
out

,seq1,seq2,seq1_pos,seq2_pos,match
aln_pos,,,,,
1,G,-,1,<NA>,gap
2,S,-,2,<NA>,gap
3,H,-,3,<NA>,gap
4,S,S,4,1,identical
5,M,M,5,2,identical
6,R,R,6,3,identical
7,Y,Y,7,4,identical
8,F,F,8,5,identical
9,Y,F,9,6,similar


## Statistics

In [24]:
out.statistics

sequences             2
mode             global
score              43.0
length               51
identity       0.627451
similarity     0.647059
coverage       0.888889
matches              32
similar               1
mismatches            1
gap_columns          18
seq1_gaps            15
seq1_start            1
seq1_end             36
seq2_gaps             3
seq2_start            1
seq2_end             48
Name: statistics, dtype: object

## Print the alignment
In blocks, or on one line:

In [25]:
out.show(width=30)

seq1     1 GSHSMRYFYTSVSRPGRGEPRFIAVGYVDD 30
              |||||:|||||||||||||||||||||
seq2     1 ---SMRYFFTSVSRPGRGEPRFIAVGYVDD 27

seq1    31 TQFVRF--------------- 36
           ||||||               
seq2    28 TQFVRFDSDAASQRMEPRAPW 48



In [26]:
out.show(one_line=True)

seq1     1 GSHSMRYFYTSVSRPGRGEPRFIAVGYVDDTQFVRF--------------- 36
              |||||:|||||||||||||||||||||||||||               
seq2     1 ---SMRYFFTSVSRPGRGEPRFIAVGYVDDTQFVRFDSDAASQRMEPRAPW 48



## Names and local mode
Name the sequences with `names=` (or `SeqFrame(seq, name=...)`). `mode='local'` keeps only the aligned region.

In [27]:
loc = SeqFrame.align('YFTSVSRPG', seq2, mode='local', names=('frag', 'hla'))
loc.show(one_line=True)
loc.statistics[['identity', 'hla_start', 'hla_end']]

frag     2 FTSVSRPG 9
           ||||||||
hla      6 FTSVSRPG 13



identity     1.0
hla_start      6
hla_end       13
Name: statistics, dtype: object

In [28]:
loc

,frag,hla,frag_pos,hla_pos,match
aln_pos,,,,,
1,F,F,2,6,identical
2,T,T,3,7,identical
3,S,S,4,8,identical
4,V,V,5,9,identical
5,S,S,6,10,identical
6,R,R,7,11,identical
7,P,P,8,12,identical
8,G,G,9,13,identical


## Work with the table
`out.df` is a plain DataFrame. For example, these are the columns that aren't identical:

In [29]:
out.df[out.df['match'] != 'identical']

,seq1,seq2,seq1_pos,seq2_pos,match
aln_pos,,,,,
1,G,-,1,<NA>,gap
2,S,-,2,<NA>,gap
3,H,-,3,<NA>,gap
9,Y,F,9,6,similar
37,-,D,<NA>,34,gap
38,-,S,<NA>,35,gap
39,-,D,<NA>,36,gap
40,-,A,<NA>,37,gap
41,-,A,<NA>,38,gap


## Drop gap regions
`drop_gaps()` keeps only the columns where every sequence has a residue. `drop_gaps('seq1')` only drops columns where seq1 has a gap.

In [30]:
print(out.show())

seq1     1 GSHSMRYFYTSVSRPGRGEPRFIAVGYVDDTQFVRF--------------- 36
              |||||:|||||||||||||||||||||||||||               
seq2     1 ---SMRYFFTSVSRPGRGEPRFIAVGYVDDTQFVRFDSDAASQRMEPRAPW 48

None


In [31]:
out.drop_gaps().show(one_line=True)

seq1     4 SMRYFYTSVSRPGRGEPRFIAVGYVDDTQFVRF 36
           |||||:|||||||||||||||||||||||||||
seq2     1 SMRYFFTSVSRPGRGEPRFIAVGYVDDTQFVRF 33



In [32]:
out.drop_gaps().statistics

sequences             2
mode             global
score              None
length               33
identity       0.969697
similarity          1.0
coverage       0.969697
matches              32
similar               1
mismatches            1
gap_columns           0
seq1_gaps             0
seq1_start            4
seq1_end             36
seq2_gaps             0
seq2_start            1
seq2_end             33
Name: statistics, dtype: object

`drop_gaps()` is not a local alignment: it filters the columns of this global alignment and does not re-align, so `mode` stays `'global'` and the score is cleared. It drops every gapped column, the gaps inside (insertions, deletions) as well as the overhangs at the ends, while `mode='local'` re-aligns and can keep gaps inside the region it finds. Here all the gaps are at the ends, so both give the same table and identity.

The statistics then describe the gap-free columns: `identity` is the identity over the residues the two share. `coverage` is computed on the remaining table too, so take it from `out.statistics` instead.

In [33]:
local = SeqFrame.align(seq1, seq2, mode='local')
pd.DataFrame({'global': out.statistics, 'drop_gaps': out.drop_gaps().statistics, 'local': local.statistics}).loc[
    ['mode', 'score', 'length', 'identity', 'coverage', 'gap_columns']]

,global,drop_gaps,local
mode,global,global,local
score,43.0,None,63.0
length,51,33,33
identity,0.627451,0.969697,0.969697
coverage,0.888889,0.969697,0.969697
gap_columns,18,0,0


With a gap inside, the two differ: `drop_gaps()` removes the inserted residue, the local alignment keeps it.

In [34]:
ins = SeqFrame.align('MKVLWAALLVTF', 'KVLWGAALLV')
ins.show(one_line=True)
ins.drop_gaps().show(one_line=True)
SeqFrame.align('MKVLWAALLVTF', 'KVLWGAALLV', mode='local').show(one_line=True)

seq1     1 MKVLW-AALLVTF 12
            |||| |||||  
seq2     1 -KVLWGAALLV-- 10

seq1     2 KVLWAALLV 10
           |||||||||
seq2     1 KVLWAALLV 10

seq1     2 KVLW-AALLV 10
           |||| |||||
seq2     1 KVLWGAALLV 10



In [35]:
out.drop_gaps('seq2').show(one_line=True)

seq1     4 SMRYFYTSVSRPGRGEPRFIAVGYVDDTQFVRF--------------- 36
           |||||:|||||||||||||||||||||||||||               
seq2     1 SMRYFFTSVSRPGRGEPRFIAVGYVDDTQFVRFDSDAASQRMEPRAPW 48



## More than two sequences
Pass more sequences to `align`. Each one is aligned to the first sequence (the reference) and the results are merged.

In [36]:
cdr3 = SeqFrame.align('CASSIRSSYEQYF', 'CASSIRSAYEQYF', 'CASSLGQAYEQYF', 'CASSIRSSGYEQYF',
                      names=['tcr1', 'tcr2', 'tcr3', 'tcr4'])
cdr3.show()

tcr1     1 CASSIRSS-YEQYF 13
tcr2     1 CASSIRSA-YEQYF 13
tcr3     1 CASSLGQA-YEQYF 13
tcr4     1 CASSIRSSGYEQYF 14
           ||||:... |||||



In [37]:
cdr3

,tcr1,tcr2,tcr3,tcr4,tcr1_pos,tcr2_pos,tcr3_pos,tcr4_pos,match
aln_pos,,,,,,,,,
1,C,C,C,C,1,1,1,1,identical
2,A,A,A,A,2,2,2,2,identical
3,S,S,S,S,3,3,3,3,identical
4,S,S,S,S,4,4,4,4,identical
5,I,I,L,I,5,5,5,5,similar
6,R,R,G,R,6,6,6,6,mismatch
7,S,S,Q,S,7,7,7,7,mismatch
8,S,A,A,S,8,8,8,8,mismatch
9,-,-,-,G,<NA>,<NA>,<NA>,9,gap


In [38]:
cdr3.drop_gaps().show()

tcr1     1 CASSIRSSYEQYF 13
tcr2     1 CASSIRSAYEQYF 13
tcr3     1 CASSLGQAYEQYF 13
tcr4     1 CASSIRSSYEQYF 14
           ||||:...|||||



## Against one reference
The first sequence is the reference. `show(reference=...)` prints residues identical to the reference as `.`, so only the differences show:

In [39]:
cdr3.show(reference='tcr1')

tcr1     1 CASSIRSS-YEQYF 13
tcr2     1 .......A-..... 13
tcr3     1 ....LGQA-..... 13
tcr4     1 ........G..... 14
           ||||:... |||||



`pairwise()` gives each sequence against the reference as its own two-sequence alignment, with its own match line and statistics:

In [40]:
pairs = cdr3.pairwise()
for p in pairs.values():
    p.show()

tcr1     1 CASSIRSSYEQYF 13
           |||||||.|||||
tcr2     1 CASSIRSAYEQYF 13

tcr1     1 CASSIRSSYEQYF 13
           ||||:...|||||
tcr3     1 CASSLGQAYEQYF 13

tcr1     1 CASSIRSS-YEQYF 13
           |||||||| |||||
tcr4     1 CASSIRSSGYEQYF 14



In [41]:
pd.DataFrame({name: p.statistics for name, p in pairs.items()}).loc[
    ['identity', 'similarity', 'matches', 'mismatches', 'gap_columns']]

,tcr2,tcr3,tcr4
identity,0.923077,0.692308,0.928571
similarity,0.923077,0.769231,0.928571
matches,12,9,13
mismatches,1,4,0
gap_columns,0,0,1


Another reference: `pairwise('tcr3')` and `show(reference='tcr3')`.

In [42]:
cdr3.show(reference='tcr3')

tcr1     1 ....IRSS-..... 13
tcr2     1 ....IRS.-..... 13
tcr3     1 CASSLGQA-YEQYF 13
tcr4     1 ....IRSSG..... 14
           ||||:... |||||

